# Этап 2: Qwen/Qwen3.5-4B (`q35_4b`)

Train v2, 2000 записей, 2 эпохи, seed 1.

QLoRA: веса модели заморожены в 4 битах, обучается только добавочный слой LoRA. Среда выполнения - GPU (T4). Результаты пишутся на Drive в `mcnl/<RUN>`; после обрыва сессии ячейки запускаются заново, обучение и ответы продолжаются с места обрыва.

## 1. Среда

In [ ]:
%cd /content
!pip -q install -U transformers peft accelerate bitsandbytes
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Drive и код проекта
Код берётся из GitHub. Для закрытого репозитория - секрет `GITHUB_TOKEN` (только чтение) в панели ключей слева.

In [ ]:
import os, subprocess
os.chdir('/content')
from google.colab import drive, userdata
drive.mount('/content/drive')
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
REPO = 'github.com/xdstk/lang.cir.spice.mc12'
!rm -rf /content/repo
done = subprocess.run(['git', 'clone', '-q', '--depth', '1', f'https://x-access-token:{token}@{REPO}' if token else f'https://{REPO}', '/content/repo'],
                      capture_output=True, text=True, env={'GIT_TERMINAL_PROMPT': '0', 'PATH': '/usr/bin:/bin:/usr/local/bin'})
if done.returncode:
    raise SystemExit('Код не скачан: ' + done.stderr.replace(token or '<нет>', '***').strip())
!git -C /content/repo remote set-url origin https://{REPO}
%cd /content/repo
!git log --oneline -1

## 3. Настройки

In [ ]:
MODEL = 'Qwen/Qwen3.5-4B'
RUN = 'q35_4b'         # папка на Drive
SUBSET = 2000          # подмножество train v2: 1000, 2000 или 3000
LIMIT = 0              # 0 - всё подмножество
FORMAT = 'netlist'     # netlist или ir
EPOCHS = 2
BATCH = 2              # шаг оптимизации всегда 8 записей
ACCUM = 8 // BATCH
SEED = 1
OUT = f'/content/drive/MyDrive/mcnl/{RUN}'
!mkdir -p {OUT}
!python src/mcnl/training.py --data data/dataset/v2 --subset {SUBSET} --limit {LIMIT} --format {FORMAT} --out out/training/{RUN}

## 4. Проба
Шаблон чата и то, чему учится модель (только нетлист и знак конца реплики); затем 10 шагов обучения - память и время.

In [ ]:
import sys
sys.path.insert(0, 'src')
from transformers import AutoTokenizer
from mcnl import training
tok = AutoTokenizer.from_pretrained(MODEL)
recs = training.read_jsonl(f'out/training/{RUN}/train.jsonl')
end = training.end_of_turn(tok)
e = training.encode(tok, recs[0]['messages'], 10 ** 6, end)
print(tok.decode(e['input_ids'])[-300:])
print('УЧИТСЯ:', repr(tok.decode([t for t, l in zip(e['input_ids'], e['labels']) if l != -100])))
lens = sorted(len(training.encode(tok, r['messages'], 10 ** 6, end)['input_ids']) for r in recs)
print(f'токенов в примере: медиана {lens[len(lens) // 2]}, максимум {lens[-1]}')
!rm -rf /content/probe
!python colab/train_lora.py --model {MODEL} --data out/training/{RUN} --out /content/probe --epochs {EPOCHS} --seed {SEED} --batch {BATCH} --accum {ACCUM} --max-steps 10

## 5. Модель без обучения
30 записей dev - точка отсчёта.

In [ ]:
!python colab/predict.py --model {MODEL} --format {FORMAT} --records data/dataset/v2/dev.jsonl --limit 30 --out {OUT}/untrained_dev.jsonl --name untrained

## 6. Обучение
`loss` должен падать; `nan` или плато - ошибка.

In [ ]:
!python colab/train_lora.py --model {MODEL} --data out/training/{RUN} --out {OUT} --epochs {EPOCHS} --seed {SEED} --batch {BATCH} --accum {ACCUM}

## 7. График обучения
Сохраняется на Drive как `loss.png`.

In [ ]:
import json, matplotlib.pyplot as plt
run = json.load(open(f'{OUT}/run.json', encoding='utf-8'))
tr = [(x['step'], x['loss']) for x in run['log'] if 'loss' in x]
ev = [(x['step'], x['eval_loss']) for x in run['log'] if 'eval_loss' in x]
plt.figure(figsize=(7, 4))
plt.plot(*zip(*tr), label='обучающая выборка')
if ev:
    plt.plot(*zip(*ev), 'o-', label='dev')
plt.xlabel('шаг обучения'); plt.ylabel('потери'); plt.legend(); plt.grid(alpha=0.3)
plt.title(RUN); plt.savefig(f'{OUT}/loss.png', dpi=200, bbox_inches='tight'); plt.show()
print(f"Время обучения: {run['train_seconds'] / 60:.1f} мин (все сессии), пик памяти GPU: {run['gpu_peak_gb']:.1f} ГБ, GPU: {run['gpu']}")
print('Отброшено длинных примеров (train, dev):', run['dropped_long'])
print('Версии:', run['versions'])

## 8. Ответы обученной модели
30 записей train, весь dev (150), запросы C01.

In [ ]:
A = f'--model {MODEL} --adapter {OUT}/adapter --format {FORMAT} --name {RUN}'
!python colab/predict.py {A} --records data/dataset/v2/train.jsonl --limit 30 --out {OUT}/train30.jsonl
!python colab/predict.py {A} --records data/dataset/v2/dev.jsonl --out {OUT}/dev.jsonl
!python colab/predict.py {A} --records data/human_test/v1/paraphrase.jsonl --out {OUT}/paraphrase.jsonl

### 8б. Остальные выборки
`mc12_real`, `gen_*`, test v2 - около 800 запросов; после обрыва - ячейки 1-3 и эта.

In [ ]:
A = f'--model {MODEL} --adapter {OUT}/adapter --format {FORMAT} --name {RUN}'
!python colab/predict.py {A} --records data/dataset/mc12_real_v1/mc12_real.jsonl --out {OUT}/mc12_real.jsonl
for s in ['gen_topology', 'gen_family', 'gen_range', 'test']:
    !python colab/predict.py {A} --records data/dataset/v2/{s}.jsonl --out {OUT}/{s}.jsonl

## 9. Быстрая проверка
Совпадение строк ответа с эталоном без учёта порядка. Точная оценка - в MC12, отдельно.

In [ ]:
import sys
sys.path.insert(0, 'src')
from mcnl import training
from IPython.display import display, Markdown


def lines(text):
    return sorted(l.strip() for l in (text or '').splitlines() if l.strip())


def show(title, records, answers, n=4):
    gold = {r['id']: r for r in training.read_jsonl(records)}
    rows = training.read_jsonl(answers)
    same = sum(lines(a['output']) == lines(training.answer_for(gold[a['id']], FORMAT)) for a in rows)
    display(Markdown(f'### {title}: совпало {same} из {len(rows)}'))
    for a in rows[:n]:
        g = gold[a['id']]
        print('ЗАПРОС:', g['prompt'][:200]); print('ЭТАЛОН:', training.answer_for(g, FORMAT).replace(chr(10), ' | '))
        print('ОТВЕТ: ', a['output'].replace(chr(10), ' | ')[:300]); print()


show('Без обучения, dev', 'data/dataset/v2/dev.jsonl', f'{OUT}/untrained_dev.jsonl')
show('Обучающая выборка', 'data/dataset/v2/train.jsonl', f'{OUT}/train30.jsonl')
show('Обученная модель, dev', 'data/dataset/v2/dev.jsonl', f'{OUT}/dev.jsonl')
show('Запросы C01', 'data/human_test/v1/paraphrase.jsonl', f'{OUT}/paraphrase.jsonl')